# Load libraries

In [ ]:
!pip install tensorflow-text

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
import tensorflow_text as text
import tf_keras
import numpy as np

# Load & Preprocess Dataset

In [ ]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.rename(columns = {'v1':'Category', 'v2':'Message'}, inplace = True)
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.groupby('Category').describe()

Message                                                            \
           count unique                                                top   
Category                                                                     
ham         4825   4516                             Sorry, I'll call later   
spam         747    641  Please call our customer service representativ...   

               
         freq  
Category       
ham        30  
spam        4

In [ ]:
df['spam']=df['Category'].apply(lambda x: 1 if x=='spam' else 0)
df.head()

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['Message'],df['spam'], stratify=df['spam'])

In [ ]:
X_train.head()

,Message
2977,I love u 2 my little pocy bell I am sorry but ...
366,Well i know Z will take care of me. So no worr...
3869,"not that I know of, most people up here are st..."
5255,Ok... Sweet dreams...
2366,Ok try to do week end course in coimbatore.


# Importing Keras Layers

In [ ]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

# Model Creation

In [ ]:
# def get_sentence_embeding(sentences):
#     preprocessed_text = bert_preprocess(sentences)
#     return bert_encoder(preprocessed_text)['pooled_output']

## Input layer

In [ ]:
text_input = tf_keras.layers.Input(shape=(), dtype=tf.string, name='text')

## Bert layers

In [ ]:
preprocessed_text = bert_preprocess(text_input)

In [ ]:
outputs = bert_encoder(preprocessed_text)

## Neural network layers

In [ ]:
l = tf_keras.layers.Dropout(0.1, name="dropout")(outputs['pooled_output'])
l = tf_keras.layers.Dense(1, activation='sigmoid', name="output")(l)

# Construct final model

In [ ]:
model = tf_keras.Model(inputs=[text_input], outputs = [l])

In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 keras_layer (KerasLayer)    {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
                                                                                              

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Training & Evaluation

In [ ]:
model.fit(X_train, y_train, epochs=2, batch_size=32)

Epoch 1/2
131/131 [==============================] - 52s 291ms/step - loss: 0.3454 - accuracy: 0.8689
Epoch 2/2
131/131 [==============================] - 41s 310ms/step - loss: 0.2410 - accuracy: 0.8945


In [ ]:
# Convert test labels to categorical for evaluation
model.evaluate(X_test, y_test)

# Predict
y_predicted = model.predict(X_test)
y_predicted = y_predicted.flatten()
print(y_predicted)

44/44 [==============================] - 14s 300ms/step
[0.00712461 0.01538548 0.6003004  ... 0.05019268 0.29402027 0.03651034]


In [ ]:
y_predicted = np.where(y_predicted > 0.5, 1, 0)
y_predicted

array([0, 0, 1, ..., 0, 0, 0])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
accuracy_score(y_test,y_predicted)

0.8872936109117013

In [ ]:
f1_score(y_test,y_predicted)

0.34309623430962344

In [ ]:
print(classification_report(y_test,y_predicted))

              precision    recall  f1-score   support

           0       0.89      0.99      0.94      1206
           1       0.79      0.22      0.34       187

    accuracy                           0.89      1393
   macro avg       0.84      0.61      0.64      1393
weighted avg       0.88      0.89      0.86      1393



In [ ]:
y_test

,spam
94,0
5118,0
1573,1
1550,0
214,0
...,...
1001,0
4836,0
4056,0
2830,1


In [ ]:
y_predicted

array([0, 0, 1, ..., 0, 0, 0])